# TextZoom external test for the whiteboard super-resolution model

This notebook tests the trained whiteboard stroke-restoration model on **TextZoom**, a public benchmark containing real low-resolution/high-resolution word pairs and text labels. TextZoom is not used for training here.

The main questions are:

1. Does restoration move the enlarged LR image closer to the real HR image (PSNR/SSIM)?
2. Does PaddleOCR read more words correctly after restoration (CER/exact accuracy)?
3. Does the result transfer beyond handwritten whiteboards?

> **Domain warning:** TextZoom mostly contains printed scene text, not handwritten whiteboard text. Treat this as an external generalisation test, not a replacement for the project's own whiteboard test set.

## 1. Download TextZoom once

Download the official LMDB dataset from the Google Drive link in the [TextZoom repository](https://github.com/WenjiaWang0312/TextZoom#textzoom-dataset-allocated-by-size). Extract it so the folders look like:

```text
whiteboard-ocr/data/textzoom/
  test/
    easy/data.mdb
    easy/lock.mdb
    medium/data.mdb
    medium/lock.mdb
    hard/data.mdb
    hard/lock.mdb
```

The **hard** subset is the most relevant starting point for small/distant text.

In [1]:
import os
os.environ.setdefault('PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK', 'True')

from pathlib import Path
from io import BytesIO
import csv
import importlib.util

import cv2
import jiwer
import lmdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from paddleocr import TextRecognition

def find_repo(start=Path.cwd()):
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / 'scripts' / 'train_accepted_pairs.py').exists():
            return folder
    raise FileNotFoundError('Open this notebook from inside the whiteboard-ocr project')

REPO = find_repo()
TEXTZOOM_ROOT = REPO / 'data' / 'textzoom'
MODEL_PATH = REPO / 'models' / 'superres' / 'accepted_pair_restorer.pt'
OUTPUT_DIR = REPO / 'outputs' / 'textzoom_external_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBSETS = ['hard']          # change to ['easy', 'medium', 'hard'] for the complete test set
MAX_SAMPLES = 0             # 0 = every sample; use 50 for a quick check
RUN_OCR = True
VISUAL_EXAMPLES = 8
SEED = 20261018

print('TextZoom:', TEXTZOOM_ROOT)
print('Model:', MODEL_PATH)

TextZoom: <LOCAL_ASSIGNMENT>/whiteboard-ocr/data/textzoom
Model: <LOCAL_ASSIGNMENT>/whiteboard-ocr/models/superres/accepted_pair_restorer.pt


## 2. Read the official LMDB files

Each row contains a real LR word image, its paired HR image and the text label.

In [2]:
class TextZoomLMDB:
    def __init__(self, folder, subset):
        self.folder = Path(folder)
        self.subset = subset
        if not (self.folder / 'data.mdb').exists():
            raise FileNotFoundError(
                f'Missing {self.folder / "data.mdb"}. Download and extract TextZoom as shown above.'
            )
        self.env = lmdb.open(
            str(self.folder), readonly=True, lock=False, readahead=False,
            meminit=False, max_readers=1
        )
        with self.env.begin(write=False) as transaction:
            self.count = int(transaction.get(b'num-samples'))

    def __len__(self):
        return self.count

    @staticmethod
    def decode(value):
        if value is None:
            raise KeyError('Image key is missing from the LMDB file')
        return np.asarray(Image.open(BytesIO(value)).convert('RGB'))

    def __getitem__(self, zero_based_index):
        index = zero_based_index + 1
        with self.env.begin(write=False) as transaction:
            suffix = f'{index:09d}'.encode()
            label = transaction.get(b'label-' + suffix)
            lr = transaction.get(b'image_lr-' + suffix)
            hr = transaction.get(b'image_hr-' + suffix)
        return {
            'subset': self.subset,
            'index': index,
            'label': label.decode('utf-8') if label else '',
            'lr': self.decode(lr),
            'hr': self.decode(hr),
        }

datasets = {}
for subset in SUBSETS:
    folder = TEXTZOOM_ROOT / 'test' / subset
    datasets[subset] = TextZoomLMDB(folder, subset)
    print(f'{subset}: {len(datasets[subset])} pairs')

hard: 1343 pairs


In [3]:
sample = datasets[SUBSETS[0]][0]
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(sample['lr']); axes[0].set_title(f"Real LR: {sample['lr'].shape[1]}x{sample['lr'].shape[0]}")
axes[1].imshow(sample['hr']); axes[1].set_title(f"Real HR: {sample['hr'].shape[1]}x{sample['hr'].shape[0]}")
for axis in axes: axis.axis('off')
plt.suptitle(f"Label: {sample['label']}")
plt.tight_layout(); plt.show()

<Figure size 1200x300 with 2 Axes>

## 3. Load the whiteboard model

The TextZoom LR image is enlarged to the HR dimensions using bicubic interpolation first. The current model then performs the same normalised-grey stroke-restoration step used in Section 6 of `super_resolution_trained.ipynb`.

In [4]:
trainer_path = REPO / 'scripts' / 'train_accepted_pairs.py'
spec = importlib.util.spec_from_file_location('accepted_pair_trainer_textzoom', trainer_path)
pair_trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pair_trainer)
pair_data = pair_trainer.pair_data
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = pair_trainer.load_model(MODEL_PATH, DEVICE)
print('Inference device:', DEVICE)

def restore_textzoom(lr_rgb, hr_size):
    width, height = hr_size
    bicubic = cv2.resize(lr_rgb, (width, height), interpolation=cv2.INTER_CUBIC)
    background = pair_data.board_background(bicubic)
    normalised = pair_data.to_grey3(pair_data.normalise_board(bicubic))
    restored_normalised = pair_trainer.restore_full(model, normalised, DEVICE)
    restored = pair_data.denormalise_grey(restored_normalised, background)
    return bicubic, restored

def grey01(rgb):
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0

def psnr(image, target):
    mse = float(np.mean((image.astype(np.float64) - target.astype(np.float64)) ** 2))
    return float(-10 * np.log10(max(mse, 1e-10)))

def ssim(image, target):
    # Standard local SSIM on greyscale images, averaged over the image.
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    mu_x = cv2.GaussianBlur(image, (11, 11), 1.5)
    mu_y = cv2.GaussianBlur(target, (11, 11), 1.5)
    sigma_x = cv2.GaussianBlur(image * image, (11, 11), 1.5) - mu_x * mu_x
    sigma_y = cv2.GaussianBlur(target * target, (11, 11), 1.5) - mu_y * mu_y
    sigma_xy = cv2.GaussianBlur(image * target, (11, 11), 1.5) - mu_x * mu_y
    score = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / (
        (mu_x * mu_x + mu_y * mu_y + c1) * (sigma_x + sigma_y + c2)
    )
    return float(np.mean(score))

def normalise_text(value):
    return ' '.join(str(value).lower().split())

def cer(reference, prediction):
    return min(float(jiwer.cer(normalise_text(reference), normalise_text(prediction) or ' ')), 1.0)

Inference device: mps


## 4. Evaluate without training

The pass/fail comparison is **restored versus bicubic**. HR is the reference target and also shows the best OCR result available from the dataset.

In [5]:
recogniser = TextRecognition(model_name='PP-OCRv6_medium_rec') if RUN_OCR else None
rng = np.random.default_rng(SEED)
rows = []

def recognise(rgb):
    # PaddleOCR accepts an OpenCV-style BGR array.
    result = recogniser.predict([cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)])[0]
    return str(result['rec_text']), float(result['rec_score'])

for subset, dataset in datasets.items():
    indices = np.arange(len(dataset))
    if MAX_SAMPLES and MAX_SAMPLES < len(indices):
        indices = np.sort(rng.choice(indices, MAX_SAMPLES, replace=False))
    print(f'Evaluating {len(indices)} {subset} pairs...')
    for position, index in enumerate(indices, start=1):
        item = dataset[int(index)]
        hr_h, hr_w = item['hr'].shape[:2]
        bicubic, restored = restore_textzoom(item['lr'], (hr_w, hr_h))
        hr_g, bicubic_g, restored_g = map(grey01, [item['hr'], bicubic, restored])
        row = {
            'subset': subset, 'index': item['index'], 'label': item['label'],
            'lr_width': item['lr'].shape[1], 'lr_height': item['lr'].shape[0],
            'hr_width': hr_w, 'hr_height': hr_h,
            'bicubic_psnr': psnr(bicubic_g, hr_g),
            'restored_psnr': psnr(restored_g, hr_g),
            'bicubic_ssim': ssim(bicubic_g, hr_g),
            'restored_ssim': ssim(restored_g, hr_g),
        }
        if RUN_OCR:
            for name, image in [('lr', item['lr']), ('bicubic', bicubic), ('restored', restored), ('hr', item['hr'])]:
                text, confidence = recognise(image)
                row[f'{name}_text'] = text
                row[f'{name}_confidence'] = confidence
                row[f'{name}_cer'] = cer(item['label'], text)
                row[f'{name}_exact'] = int(normalise_text(item['label']) == normalise_text(text))
        rows.append(row)
        if position % 100 == 0 or position == len(indices):
            print(f'  {position}/{len(indices)}')

results = pd.DataFrame(rows)
results['psnr_gain'] = results.restored_psnr - results.bicubic_psnr
results['ssim_gain'] = results.restored_ssim - results.bicubic_ssim
if RUN_OCR:
    results['cer_gain'] = results.bicubic_cer - results.restored_cer
results_path = OUTPUT_DIR / 'textzoom_results.csv'
results.to_csv(results_path, index=False)
print('Saved:', results_path)

<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


Evaluating 1343 hard pairs...
  100/1343
  200/1343
  300/1343
  400/1343
  500/1343
  600/1343
  700/1343
  800/1343
  900/1343
  1000/1343
  1100/1343
  1200/1343
  1300/1343
  1343/1343
Saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/textzoom_external_test/textzoom_results.csv


In [6]:
summary_rows = []
for subset, group in results.groupby('subset'):
    record = {
        'subset': subset, 'pairs': len(group),
        'bicubic PSNR': group.bicubic_psnr.mean(),
        'restored PSNR': group.restored_psnr.mean(),
        'PSNR gain': group.psnr_gain.mean(),
        'bicubic SSIM': group.bicubic_ssim.mean(),
        'restored SSIM': group.restored_ssim.mean(),
        'SSIM gain': group.ssim_gain.mean(),
    }
    if RUN_OCR:
        record.update({
            'LR CER': group.lr_cer.mean(),
            'bicubic CER': group.bicubic_cer.mean(),
            'restored CER': group.restored_cer.mean(),
            'HR CER': group.hr_cer.mean(),
            'bicubic exact accuracy': group.bicubic_exact.mean(),
            'restored exact accuracy': group.restored_exact.mean(),
            'HR exact accuracy': group.hr_exact.mean(),
            'CER improved / unchanged / worsened': (
                f"{int((group.cer_gain > 0).sum())} / "
                f"{int((group.cer_gain == 0).sum())} / "
                f"{int((group.cer_gain < 0).sum())}"
            ),
        })
    summary_rows.append(record)
summary = pd.DataFrame(summary_rows).set_index('subset')
display(summary.round(4))

if RUN_OCR:
    for subset, row in summary.iterrows():
        gain = row['bicubic CER'] - row['restored CER']
        verdict = 'HELPS OCR' if gain > 0 else 'DOES NOT HELP OCR'
        print(f'{subset}: CER gain {gain:+.4f} -> {verdict}')

,pairs,bicubic PSNR,restored PSNR,PSNR gain,bicubic SSIM,restored SSIM,SSIM gain,LR CER,bicubic CER,restored CER,HR CER,bicubic exact accuracy,restored exact accuracy,HR exact accuracy,CER improved / unchanged / worsened
subset,,,,,,,,,,,,,,,
hard,1343,20.6553,20.103,-0.5524,0.6117,0.5875,-0.0243,0.3693,0.3693,0.3994,0.0916,0.4557,0.3999,0.8198,119 / 932 / 292


hard: CER gain -0.0302 -> DOES NOT HELP OCR


## 5. Visual comparison

Examples are selected from both helpful and harmful cases so the figure does not hide failures.

In [7]:
ranking_column = 'cer_gain' if RUN_OCR else 'psnr_gain'
ordered = results.sort_values(ranking_column)
n_each = max(1, VISUAL_EXAMPLES // 2)
chosen = pd.concat([ordered.head(n_each), ordered.tail(n_each)]).drop_duplicates(['subset', 'index'])

fig, axes = plt.subplots(len(chosen), 4, figsize=(16, max(3, 2.3 * len(chosen))))
if len(chosen) == 1:
    axes = axes[None, :]
for row_number, (_, result) in enumerate(chosen.iterrows()):
    item = datasets[result['subset']][int(result['index']) - 1]
    hr_h, hr_w = item['hr'].shape[:2]
    bicubic, restored = restore_textzoom(item['lr'], (hr_w, hr_h))
    versions = [('Real LR', item['lr']), ('Bicubic', bicubic), ('Restored', restored), ('Real HR', item['hr'])]
    for column, (name, image) in enumerate(versions):
        axes[row_number, column].imshow(image)
        title = name
        if column == 0:
            title += f" | {result['subset']} | label: {result['label']}"
        if RUN_OCR and name.lower().replace('real ', '') in ('lr', 'bicubic', 'restored', 'hr'):
            key = name.lower().replace('real ', '')
            title += f"\nOCR: {result[f'{key}_text']}"
        axes[row_number, column].set_title(title, fontsize=9)
        axes[row_number, column].axis('off')
plt.tight_layout()
figure_path = OUTPUT_DIR / 'textzoom_visual_comparison.png'
plt.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)

/var/folders/xl/0nmgmb4d27v2cp2tghhvpkg40000gn/T/ipykernel_42255/644010583.py:24: UserWarning: Glyph 27665 (\N{CJK UNIFIED IDEOGRAPH-6C11}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/xl/0nmgmb4d27v2cp2tghhvpkg40000gn/T/ipykernel_42255/644010583.py:26: UserWarning: Glyph 27665 (\N{CJK UNIFIED IDEOGRAPH-6C11}) missing from font(s) DejaVu Sans.
  plt.savefig(figure_path, dpi=180, bbox_inches='tight')
<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/IPython/core/pylabtools.py:158: UserWarning: Glyph 27665 (\N{CJK UNIFIED IDEOGRAPH-6C11}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)


<Figure size 1600x1840 with 32 Axes>

Saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/textzoom_external_test/textzoom_visual_comparison.png


## 6. How to interpret the result

The model passes this external test only if:

- restored CER is lower than bicubic CER;
- restored exact-word accuracy is higher than bicubic accuracy;
- more individual words improve than worsen; and
- the visual examples do not show invented, merged or broken character strokes.

A failure here does not automatically invalidate a whiteboard-specific model because TextZoom is a different domain. Report TextZoom and the project's unseen-whiteboard test separately.

## 7. Controlled experiment: does more paired training data help?

This trains the **same small `StrokeRestorer` architecture** on TextZoom's 17,367 `train1 + train2` pairs. It does **not** train TSRN. This isolates the value of a much larger paired dataset before we decide whether to implement a larger text-specific architecture.

The official `test/easy`, `test/medium`, and `test/hard` folders remain untouched. A seeded 1,000-pair portion of the training folders is used only for early stopping. The new checkpoint is saved separately and does not overwrite the whiteboard model.

In [8]:
import subprocess, sys

TEXTZOOM_MODEL_PATH = REPO / 'models' / 'superres' / 'textzoom_stroke_restorer.pt'
command = [
    sys.executable, str(REPO / 'scripts' / 'train_textzoom_restorer.py'),
    '--epochs', '20',
    '--batch-size', '64',
    '--patience', '5',
    '--output', str(TEXTZOOM_MODEL_PATH),
]
print('Training a separate TextZoom checkpoint. This can take a while.')
subprocess.run(command, check=True)
print('Saved:', TEXTZOOM_MODEL_PATH)

Training a separate TextZoom checkpoint. This can take a while.
TextZoom train pairs: 16,367; internal validation pairs: 1,000
Official TextZoom test subsets are untouched.
device: mps
epoch 01: train=0.0801 val input=21.99dB output=22.08dB gain=+0.08dB * best
epoch 02: train=0.0790 val input=21.99dB output=22.13dB gain=+0.14dB * best
epoch 03: train=0.0784 val input=21.99dB output=22.14dB gain=+0.15dB * best
epoch 04: train=0.0780 val input=21.99dB output=22.14dB gain=+0.14dB
epoch 05: train=0.0778 val input=21.99dB output=22.19dB gain=+0.20dB * best
epoch 06: train=0.0775 val input=21.99dB output=22.22dB gain=+0.22dB * best
epoch 07: train=0.0773 val input=21.99dB output=22.21dB gain=+0.21dB
epoch 08: train=0.0772 val input=21.99dB output=22.22dB gain=+0.22dB
epoch 09: train=0.0769 val input=21.99dB output=22.22dB gain=+0.22dB
epoch 10: train=0.0766 val input=21.99dB output=22.28dB gain=+0.29dB * best
epoch 11: train=0.0766 val input=21.99dB output=22.26dB gain=+0.27dB
epoch 12: trai

## 8. Compare the newly trained model on the untouched test set

This adds one fair comparison column: **TextZoom-trained model**. The important comparison is against bicubic and the earlier whiteboard-trained model on exactly the same test samples. Lower CER and higher exact-word accuracy are better.

In [9]:
checkpoint = torch.load(TEXTZOOM_MODEL_PATH, map_location='cpu', weights_only=False)
textzoom_model = pair_trainer.StrokeRestorer()
textzoom_model.load_state_dict(checkpoint['model'])
textzoom_model = textzoom_model.to(DEVICE).eval()
print({key: checkpoint[key] for key in ['epoch', 'train_pairs', 'val_pairs', 'val_gain']})

def restore_with_textzoom_model(lr_rgb, hr_size):
    width, height = hr_size
    bicubic = cv2.resize(lr_rgb, (width, height), interpolation=cv2.INTER_CUBIC)
    model_input = bicubic.astype(np.float32) / 255.0
    output = pair_trainer.restore_full(textzoom_model, model_input, DEVICE)
    return np.clip(np.rint(output * 255.0), 0, 255).astype(np.uint8)

textzoom_rows = []
for subset, dataset in datasets.items():
    tested_indices = results.loc[results.subset == subset, 'index'].astype(int).tolist()
    print(f'Evaluating trained model on {len(tested_indices)} untouched {subset} pairs...')
    for position, one_based_index in enumerate(tested_indices, start=1):
        item = dataset[one_based_index - 1]
        hr_h, hr_w = item['hr'].shape[:2]
        restored_tz = restore_with_textzoom_model(item['lr'], (hr_w, hr_h))
        row = {
            'subset': subset, 'index': one_based_index,
            'textzoom_model_psnr': psnr(grey01(restored_tz), grey01(item['hr'])),
            'textzoom_model_ssim': ssim(grey01(restored_tz), grey01(item['hr'])),
        }
        if RUN_OCR:
            text, confidence = recognise(restored_tz)
            row.update({
                'textzoom_model_text': text,
                'textzoom_model_confidence': confidence,
                'textzoom_model_cer': cer(item['label'], text),
                'textzoom_model_exact': int(normalise_text(item['label']) == normalise_text(text)),
            })
        textzoom_rows.append(row)
        if position % 100 == 0 or position == len(tested_indices):
            print(f'  {position}/{len(tested_indices)}')

textzoom_results = pd.DataFrame(textzoom_rows)
comparison = results.merge(textzoom_results, on=['subset', 'index'], validate='one_to_one')
comparison.to_csv(OUTPUT_DIR / 'textzoom_model_comparison.csv', index=False)

summary_columns = {
    'pairs': len(comparison),
    'bicubic PSNR': comparison.bicubic_psnr.mean(),
    'whiteboard-model PSNR': comparison.restored_psnr.mean(),
    'TextZoom-model PSNR': comparison.textzoom_model_psnr.mean(),
    'bicubic SSIM': comparison.bicubic_ssim.mean(),
    'whiteboard-model SSIM': comparison.restored_ssim.mean(),
    'TextZoom-model SSIM': comparison.textzoom_model_ssim.mean(),
}
if RUN_OCR:
    summary_columns.update({
        'bicubic CER': comparison.bicubic_cer.mean(),
        'whiteboard-model CER': comparison.restored_cer.mean(),
        'TextZoom-model CER': comparison.textzoom_model_cer.mean(),
        'HR CER': comparison.hr_cer.mean(),
        'bicubic exact': comparison.bicubic_exact.mean(),
        'whiteboard-model exact': comparison.restored_exact.mean(),
        'TextZoom-model exact': comparison.textzoom_model_exact.mean(),
        'HR exact': comparison.hr_exact.mean(),
    })
display(pd.DataFrame([summary_columns], index=['untouched test']).round(4))
print('Saved:', OUTPUT_DIR / 'textzoom_model_comparison.csv')

{'epoch': 10, 'train_pairs': 16367, 'val_pairs': 1000, 'val_gain': 0.2850344409942629}
Evaluating trained model on 1343 untouched hard pairs...
  100/1343
  200/1343
  300/1343
  400/1343
  500/1343
  600/1343
  700/1343
  800/1343
  900/1343
  1000/1343
  1100/1343
  1200/1343
  1300/1343
  1343/1343


,pairs,bicubic PSNR,whiteboard-model PSNR,TextZoom-model PSNR,bicubic SSIM,whiteboard-model SSIM,TextZoom-model SSIM,bicubic CER,whiteboard-model CER,TextZoom-model CER,HR CER,bicubic exact,whiteboard-model exact,TextZoom-model exact,HR exact
untouched test,1343,20.6553,20.103,20.832,0.6117,0.5875,0.6371,0.3693,0.3994,0.3997,0.0916,0.4557,0.3999,0.4118,0.8198


Saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/textzoom_external_test/textzoom_model_comparison.csv


## 9. Next experiment: deeper residual, stroke-aware CNN

The large dataset improved PSNR/SSIM but not OCR. This controlled follow-up changes the model and loss while keeping the same TextZoom training split:

- six residual blocks with 64 feature channels;
- edge/stroke-weighted L1 plus the existing gradient loss;
- consistent `128x32` geometry in training and testing;
- checkpoint selection by PaddleOCR CER on a fixed 256-image validation subset.

The checkpoint is separate from both earlier models. The official test folders are still untouched during training.

In [10]:
RESIDUAL_MODEL_PATH = REPO / 'models' / 'superres' / 'textzoom_residual_stroke.pt'
residual_command = [
    sys.executable, str(REPO / 'scripts' / 'train_textzoom_residual.py'),
    '--epochs', '20',
    '--batch-size', '64',
    '--blocks', '6',
    '--channels', '64',
    '--stroke-weight', '4.0',
    '--ocr-val-size', '256',
    '--patience', '5',
    '--output', str(RESIDUAL_MODEL_PATH),
]
print('Training deeper residual/stroke-aware model...')
subprocess.run(residual_command, check=True)
print('Saved:', RESIDUAL_MODEL_PATH)

Training deeper residual/stroke-aware model...
TextZoom train pairs: 16,367; validation: 1,000; OCR validation: 256
Official TextZoom test subsets are untouched.


<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


device: mps; baseline OCR CER=0.4071, exact=0.4453
architecture: 6 residual blocks, 64 channels
loss: edge-weighted L1 (strength 4.0) + 0.2 x gradient
epoch 01: train=0.0863 PSNR gain=-0.01dB OCR CER=0.4130 exact=0.4062 * best OCR
epoch 02: train=0.0852 PSNR gain=+0.12dB OCR CER=0.4178 exact=0.3984
epoch 03: train=0.0846 PSNR gain=+0.16dB OCR CER=0.4135 exact=0.4023
epoch 04: train=0.0842 PSNR gain=+0.17dB OCR CER=0.4120 exact=0.3945 * best OCR
epoch 05: train=0.0839 PSNR gain=+0.17dB OCR CER=0.4156 exact=0.3984
epoch 06: train=0.0836 PSNR gain=+0.24dB OCR CER=0.4228 exact=0.3945
epoch 07: train=0.0834 PSNR gain=+0.25dB OCR CER=0.4249 exact=0.3945
epoch 08: train=0.0832 PSNR gain=+0.25dB OCR CER=0.4347 exact=0.3828
epoch 09: train=0.0831 PSNR gain=+0.29dB OCR CER=0.4280 exact=0.3945
early stopping after 5 epochs without OCR improvement
saved best OCR checkpoint from epoch 4: <LOCAL_ASSIGNMENT>/whiteboard-ocr/models/superres/textzoom_residual_stroke.pt
Saved: <LOCAL_ASSIGNMENT>/whiteboa

## 10. Fair test at the same `128x32` geometry

This re-evaluates the shallow TextZoom model and the new residual model with exactly the geometry used during training. It compares them against the standardised LR input and standardised real HR target.

In [11]:
residual_script = REPO / 'scripts' / 'train_textzoom_residual.py'
residual_spec = importlib.util.spec_from_file_location('textzoom_residual_module', residual_script)
residual_module = importlib.util.module_from_spec(residual_spec)
residual_spec.loader.exec_module(residual_module)
residual_checkpoint = torch.load(RESIDUAL_MODEL_PATH, map_location='cpu', weights_only=False)
residual_model = residual_module.ResidualStrokeRestorer(
    residual_checkpoint['blocks'], residual_checkpoint['channels']
).to(DEVICE)
residual_model.load_state_dict(residual_checkpoint['model'])
residual_model.eval()
print({key: residual_checkpoint[key] for key in [
    'epoch', 'blocks', 'channels', 'val_gain', 'val_ocr_cer', 'val_ocr_exact'
]})

def resize_textzoom(rgb):
    return np.asarray(Image.fromarray(rgb).resize((128, 32), Image.Resampling.BICUBIC))

def run_torch_restorer(torch_model, rgb):
    array = rgb.astype(np.float32) / 255.0
    with torch.no_grad():
        tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        output = torch_model(tensor)[0].permute(1, 2, 0).cpu().numpy()
    return np.clip(np.rint(output * 255.0), 0, 255).astype(np.uint8)

standard_rows = []
ocr_images = {'lr128': [], 'shallow': [], 'residual': [], 'hr128': []}
for subset, dataset in datasets.items():
    tested_indices = results.loc[results.subset == subset, 'index'].astype(int).tolist()
    print(f'Preparing {len(tested_indices)} standardised {subset} samples...')
    for position, one_based_index in enumerate(tested_indices, start=1):
        item = dataset[one_based_index - 1]
        lr128, hr128 = resize_textzoom(item['lr']), resize_textzoom(item['hr'])
        shallow = run_torch_restorer(textzoom_model, lr128)
        residual = run_torch_restorer(residual_model, lr128)
        row = {'subset': subset, 'index': one_based_index, 'label': item['label']}
        for name, image in [('lr128', lr128), ('shallow', shallow), ('residual', residual)]:
            row[f'{name}_psnr'] = psnr(grey01(image), grey01(hr128))
            row[f'{name}_ssim'] = ssim(grey01(image), grey01(hr128))
        standard_rows.append(row)
        if RUN_OCR:
            for name, image in [('lr128', lr128), ('shallow', shallow), ('residual', residual), ('hr128', hr128)]:
                ocr_images[name].append(cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        if position % 200 == 0 or position == len(tested_indices):
            print(f'  {position}/{len(tested_indices)}')

standard_results = pd.DataFrame(standard_rows)
if RUN_OCR:
    for name, images in ocr_images.items():
        print(f'OCR: {name}...')
        predictions = recogniser.predict(images)
        texts = [str(prediction['rec_text']) for prediction in predictions]
        standard_results[f'{name}_text'] = texts
        standard_results[f'{name}_cer'] = [cer(label, text) for label, text in zip(standard_results.label, texts)]
        standard_results[f'{name}_exact'] = [
            int(normalise_text(label) == normalise_text(text))
            for label, text in zip(standard_results.label, texts)
        ]

standard_path = OUTPUT_DIR / 'textzoom_residual_comparison_128x32.csv'
standard_results.to_csv(standard_path, index=False)
summary128 = {
    'pairs': len(standard_results),
    'LR PSNR': standard_results.lr128_psnr.mean(),
    'shallow PSNR': standard_results.shallow_psnr.mean(),
    'residual PSNR': standard_results.residual_psnr.mean(),
    'LR SSIM': standard_results.lr128_ssim.mean(),
    'shallow SSIM': standard_results.shallow_ssim.mean(),
    'residual SSIM': standard_results.residual_ssim.mean(),
}
if RUN_OCR:
    for name in ['lr128', 'shallow', 'residual', 'hr128']:
        summary128[f'{name} CER'] = standard_results[f'{name}_cer'].mean()
        summary128[f'{name} exact'] = standard_results[f'{name}_exact'].mean()
display(pd.DataFrame([summary128], index=['hard 128x32']).round(4))
if RUN_OCR:
    gain = standard_results.lr128_cer - standard_results.residual_cer
    print('Residual CER improved / unchanged / worsened:',
          int((gain > 0).sum()), '/', int((gain == 0).sum()), '/', int((gain < 0).sum()))
print('Saved:', standard_path)

{'epoch': 4, 'blocks': 6, 'channels': 64, 'val_gain': 0.1719737272262556, 'val_ocr_cer': 0.4119897822067423, 'val_ocr_exact': 0.39453125}
Preparing 1343 standardised hard samples...
  200/1343
  400/1343
  600/1343
  800/1343
  1000/1343
  1200/1343
  1343/1343
OCR: lr128...
OCR: shallow...
OCR: residual...
OCR: hr128...


,pairs,LR PSNR,shallow PSNR,residual PSNR,LR SSIM,shallow SSIM,residual SSIM,lr128 CER,lr128 exact,shallow CER,shallow exact,residual CER,residual exact,hr128 CER,hr128 exact
hard 128x32,1343,20.7603,20.9992,20.9849,0.6226,0.6584,0.6411,0.4283,0.3902,0.4547,0.3418,0.4406,0.3723,0.1048,0.8004


Residual CER improved / unchanged / worsened: 95 / 1091 / 157
Saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/textzoom_external_test/textzoom_residual_comparison_128x32.csv


## 11. Official pretrained text-SR benchmark

This evaluates PaddleOCR's official **Text Gestalt TSRN variant** and pretrained TextZoom checkpoint already stored in this project. It is a research text-super-resolution model using TSRN, spatial transformation/alignment, and text-oriented training. It is related to—but not identical to—the original minimal TSRN release.

The official geometry is used exactly: `16x64` LR input to `32x128` SR output. For a controlled comparison, bicubic, the shallow CNN and our residual CNN all receive the same `16x64` information. No training occurs in this section.

In [12]:
import sys
import paddle

PADDLEOCR_SR_SOURCE = REPO.parent / 'models' / 'PaddleOCR-text-sr'
TSRN_WEIGHTS = REPO / 'models' / 'superres' / 'text_gestalt' / 'sr_tsrn_transformer_strock_train' / 'best_accuracy.pdparams'
assert PADDLEOCR_SR_SOURCE.is_dir(), PADDLEOCR_SR_SOURCE
assert TSRN_WEIGHTS.exists(), TSRN_WEIGHTS
sys.path.insert(0, str(PADDLEOCR_SR_SOURCE))
from ppocr.modeling.architectures import build_model as build_text_sr_model

tsrn_config = dict(
    model_type='sr', algorithm='Gestalt',
    Transform=dict(name='TSRN', STN=True, infer_mode=True),
)
official_tsrn = build_text_sr_model(tsrn_config)
official_tsrn.set_state_dict(paddle.load(str(TSRN_WEIGHTS)))
official_tsrn.eval()
print(f'Loaded official TSRN variant: {TSRN_WEIGHTS.name} ({TSRN_WEIGHTS.stat().st_size / 1e6:.1f} MB)')

official_samples = []
for subset, dataset in datasets.items():
    tested_indices = results.loc[results.subset == subset, 'index'].astype(int).tolist()
    for one_based_index in tested_indices:
        item = dataset[one_based_index - 1]
        lr64 = cv2.resize(item['lr'], (64, 16), interpolation=cv2.INTER_AREA)
        bicubic128 = cv2.resize(lr64, (128, 32), interpolation=cv2.INTER_CUBIC)
        hr128 = cv2.resize(item['hr'], (128, 32), interpolation=cv2.INTER_CUBIC)
        official_samples.append({
            'subset': subset, 'index': one_based_index, 'label': item['label'],
            'lr64': lr64, 'bicubic': bicubic128, 'hr': hr128,
            'shallow': run_torch_restorer(textzoom_model, bicubic128),
            'residual': run_torch_restorer(residual_model, bicubic128),
        })

TSRN_BATCH = 32
for start in range(0, len(official_samples), TSRN_BATCH):
    batch = official_samples[start:start + TSRN_BATCH]
    array = np.stack([sample['lr64'].transpose(2, 0, 1) for sample in batch]).astype('float32') / 255.0
    with paddle.no_grad():
        output = official_tsrn(paddle.to_tensor(array))['sr_img'].numpy()
    output = np.clip(np.rint(output.transpose(0, 2, 3, 1) * 255.0), 0, 255).astype(np.uint8)
    for sample, image in zip(batch, output):
        sample['tsrn'] = image
    if start % 320 == 0 or start + len(batch) == len(official_samples):
        print(f'TSRN {start + len(batch)}/{len(official_samples)}')

Skipping import of the encryption module.
Loaded official TSRN variant: best_accuracy.pdparams (271.0 MB)
TSRN 32/1343
TSRN 352/1343
TSRN 672/1343
TSRN 992/1343
TSRN 1312/1343
TSRN 1343/1343


In [13]:
official_rows = []
methods = ['bicubic', 'shallow', 'residual', 'tsrn', 'hr']
for sample in official_samples:
    row = {key: sample[key] for key in ['subset', 'index', 'label']}
    for method in methods[:-1]:
        row[f'{method}_psnr'] = psnr(grey01(sample[method]), grey01(sample['hr']))
        row[f'{method}_ssim'] = ssim(grey01(sample[method]), grey01(sample['hr']))
    official_rows.append(row)
official_results = pd.DataFrame(official_rows)

if RUN_OCR:
    for method in methods:
        print(f'OCR: {method}...')
        bgr_images = [cv2.cvtColor(sample[method], cv2.COLOR_RGB2BGR) for sample in official_samples]
        predictions = recogniser.predict(bgr_images)
        texts = [str(prediction['rec_text']) for prediction in predictions]
        official_results[f'{method}_text'] = texts
        official_results[f'{method}_cer'] = [cer(label, text) for label, text in zip(official_results.label, texts)]
        official_results[f'{method}_exact'] = [
            int(normalise_text(label) == normalise_text(text))
            for label, text in zip(official_results.label, texts)
        ]

official_path = OUTPUT_DIR / 'official_tsrn_comparison.csv'
official_results.to_csv(official_path, index=False)
summary_official = {'pairs': len(official_results)}
for method in methods[:-1]:
    summary_official[f'{method} PSNR'] = official_results[f'{method}_psnr'].mean()
    summary_official[f'{method} SSIM'] = official_results[f'{method}_ssim'].mean()
if RUN_OCR:
    for method in methods:
        summary_official[f'{method} CER'] = official_results[f'{method}_cer'].mean()
        summary_official[f'{method} exact'] = official_results[f'{method}_exact'].mean()
display(pd.DataFrame([summary_official], index=['TextZoom hard official geometry']).round(4))
if RUN_OCR:
    gain = official_results.bicubic_cer - official_results.tsrn_cer
    print('TSRN versus bicubic — CER improved / unchanged / worsened:',
          int((gain > 0).sum()), '/', int((gain == 0).sum()), '/', int((gain < 0).sum()))
    print(f'Mean TSRN CER change: {-gain.mean():+.4f} (negative is better)')
print('Saved:', official_path)

OCR: bicubic...
OCR: shallow...
OCR: residual...
OCR: tsrn...
OCR: hr...


,pairs,bicubic PSNR,bicubic SSIM,shallow PSNR,shallow SSIM,residual PSNR,residual SSIM,tsrn PSNR,tsrn SSIM,bicubic CER,bicubic exact,shallow CER,shallow exact,residual CER,residual exact,tsrn CER,tsrn exact,hr CER,hr exact
TextZoom hard official geometry,1343,20.5161,0.6063,20.7877,0.6443,20.756,0.6262,20.123,0.6007,0.4499,0.3723,0.4612,0.3351,0.4567,0.3589,0.4442,0.3552,0.1064,0.7997


TSRN versus bicubic — CER improved / unchanged / worsened: 295 / 786 / 262
Mean TSRN CER change: -0.0056 (negative is better)
Saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/textzoom_external_test/official_tsrn_comparison.csv


## 12. Inspect where TSRN helps and hurts

The figure deliberately includes the strongest improvements and strongest regressions, preventing cherry-picking.

In [14]:
rank = official_results.bicubic_cer - official_results.tsrn_cer if RUN_OCR else official_results.tsrn_psnr - official_results.bicubic_psnr
selected_indices = list(rank.nsmallest(4).index) + list(rank.nlargest(4).index)
fig, axes = plt.subplots(len(selected_indices), 5, figsize=(16, 2.2 * len(selected_indices)))
for row_number, result_index in enumerate(selected_indices):
    sample, result = official_samples[result_index], official_results.iloc[result_index]
    for column, method in enumerate(['bicubic', 'shallow', 'residual', 'tsrn', 'hr']):
        axes[row_number, column].imshow(sample[method])
        title = method
        if RUN_OCR:
            title += f" | OCR: {result[f'{method}_text']}"
        if column == 0:
            title += f"\nlabel: {result['label']}"
        axes[row_number, column].set_title(title, fontsize=8)
        axes[row_number, column].axis('off')
plt.tight_layout()
official_figure = OUTPUT_DIR / 'official_tsrn_visual_comparison.png'
plt.savefig(official_figure, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', official_figure)

<Figure size 1600x1760 with 40 Axes>

Saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/textzoom_external_test/official_tsrn_visual_comparison.png
